In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain.agents import create_agent

In [4]:
loader = PyPDFLoader("../data/AYUSH_230620.pdf")
docs = loader.load()

In [5]:
len(docs)

1

In [9]:
spilitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = spilitter.split_documents(docs)

In [11]:
len(splitted_docs)

4

In [16]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = InMemoryVectorStore.from_documents(
    documents = splitted_docs, 
    embedding = embeddings
    )

In [21]:
same_record = vectorstore.similarity_search("Education")

In [22]:
same_record[0].page_content

'AYUSH SINGH CHANDEL\n+91-9693794566⋄Pune, Maharashtra, India\nayushsinghchandel1610@gmail.com⋄linkedin.com/in/ayushsingh⋄github.com/Ayushsingh\nEDUCA TION\nArmy Institute of T echnology , Pune Aug 2023 – Expected May 2027\nBachelor of Engineering, Electronics and Telecommunication Engineering CGPA: 8.17\nSKILLS\nLanguages C/C++, JavaScript, TypeScript, SQL\nF rontend React.js, Next.js, T ailwind CSS, Redux, Recoil\nBackend Node.js, Express.js, REST APIs, W ebSockets, Serverless\nDatabases PostgreSQL, MongoDB, Redis, Prisma, Mongoose\nDevOps A WS (EC2, S3), Docker, CI/CD, Git, T urborepo\nCS F undamentals DSA, OOPs, DBMS, Computer Networks\nEXPERIENCE\nSmart India Hackathon (Inter-College) Sep 2024 – Oct 2024\n•Led a 5-member team to buildIntellectGuard, a unified MERN stack platform centralizing trademark registration,\npatent filing, and research publishing into a single ecosystem.\n•Spearheaded project planning and UI/UX design, conducted regular agile meetings and creating highly i

### TOOLS LLM PROMPTS

In [65]:
@tool
def retriever_tool(query:str):
    """Retrieves relevant information from the vectorstore based on the query."""
    print(f"Query received: {query}")
    same_record = vectorstore.similarity_search(query)
    context = ""
    for record in same_record:
        context += record.page_content + "\n"
    return context

In [54]:
llm = ChatOllama(model="qwen3:4b")

In [55]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt="You are a helpful assistant that answers questions based on the context provided by the retriever_tool. If the answer is not found in the context, say 'I don't know'."
)

In [ ]:
query = "What are the projects and skills of Ayush Singh Chandel?"
response = agent.invoke({"messages":[{"role": "user", "content": query}]})

In [63]:
result = response["messages"][-1].content

In [64]:
print(result)

Ayush Singh Chandel has the following projects:

1. **IntellectGuard** (Smart India Hackathon, Sep 2024 – Oct 2024): A unified MERN stack platform centralizing trademark registration, patent filing, and research publishing into a single ecosystem, securing 2nd Position in the competition.  
2. **PhotoAI** (Sep 2025 – Dec 2025): A full-stack AI photo studio using Next.js and fal.ai, enabling users to train face models on personal photos and generate customized images via prompts and style packs.  
3. **PayNet** (Feb 2025 – Mar 2025): A peer-to-peer payment application using the MERN stack with MongoDB transactions to ensure atomic fund transfers and ACID compliance.  
4. **EasyDraw** (Jul 2025 – Aug 2025): A real-time collaborative whiteboard utilizing Next.js and WebSockets, allowing remote teams to draw in secure, private rooms with synchronized state broadcasting.
